# 06. Recursion to Iteration, Stack Limits and Memoization Tools

Recursion is elegant but Python has a **recursion limit** (about 1000) and no tail-call optimization. Production code and some interviews ask: "can you do this **without recursion**?"

**Learning order:** WHY LIMITS → ITERATIVE VERSIONS → EXPLICIT STACK → TRAMPOLINE → MEMO TOOLS → MUTUAL RECURSION → INTERVIEW THINKING

## 1. Why recursion hits limits

Each call creates a **stack frame**. Python limits the depth (`sys.getrecursionlimit()`, default 1000) to avoid crashing the interpreter. A linked list or a skewed tree with 10⁵ nodes would exceed it.

Options: (1) rewrite iteratively, (2) use an **explicit stack** you manage, (3) raise the limit with `sys.setrecursionlimit` (risky: deep recursion can still crash the process), (4) use a language with tail-call optimization.

In [1]:
import sys

def depth_sum(n):
    return 0 if n == 0 else n + depth_sum(n - 1)

print(sys.getrecursionlimit(), depth_sum(500))   # 1000 125250
try:
    depth_sum(10**5)
except RecursionError:
    print("RecursionError")   # RecursionError

1000 125250
RecursionError


## 2. Iterative versions of simple recursions

**Linear recursion (tail position)** → a loop with accumulators. **Two-term recursion** (Fibonacci) → two variables. **Divide in half** → a loop shrinking the range.

In [2]:
def depth_sum_iter(n):
    total = 0
    while n:
        total += n
        n -= 1
    return total

print(depth_sum_iter(10**5))   # 5000050000

5000050000


## 3. General recipe: simulate the call stack

Make an explicit **stack of pending work**. Push the initial call. Loop: pop, do the work, push the sub-calls (in **reverse** order if order matters). Example: sum of all integers in an arbitrarily nested list.

In [3]:
def nested_sum_recursive(x):
    return x if isinstance(x, int) else sum(nested_sum_recursive(i) for i in x)

def nested_sum_iterative(data):
    total, stack = 0, [data]
    while stack:
        item = stack.pop()
        if isinstance(item, int):
            total += item
        else:
            stack.extend(item)                                # defer the children
    return total

data = [1, [2, [3, 4]], 5]
print(nested_sum_recursive(data), nested_sum_iterative(data))   # 15 15

# deep nesting: recursion fails, the explicit stack does not
deep = 0
for _ in range(5000):
    deep = [deep, 1]
print(nested_sum_iterative(deep))   # 5000

15 15
5000


## 4. Trampoline (tail recursion without growing the stack)

A **trampoline** runs a loop: the function returns **another function (a thunk)** instead of calling itself, and the loop calls it. The Python stack stays flat. Useful to show you understand tail calls; rarely used in real code.

In [4]:
def trampoline(fn, *args):
    result = fn(*args)
    while callable(result):
        result = result()
    return result

def sum_to(n, acc=0):
    if n == 0:
        return acc
    return lambda: sum_to(n - 1, acc + n)          # return a thunk instead of recursing

print(trampoline(sum_to, 100000))   # 5000050000

5000050000


## 5. Memoization tools

- `functools.lru_cache(maxsize=None)` / `functools.cache` (3.9+): decorator; arguments must be **hashable**; `.cache_clear()` and `.cache_info()` exist.
- A manual `dict` memo gives control (clear it, inspect it, use tuple keys).
- For deep recursion, memoized recursion still hits the limit: use **bottom-up** tables.

In [5]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80), fib.cache_info().misses)   # 23416728348467685 81
fib.cache_clear()

def fib_bottom_up(n):                           # no recursion at all, any n
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

print(fib_bottom_up(5000) > 10**1000)   # True

23416728348467685 81
True


## 6. Mutual recursion

Two functions calling each other. Fine in small cases, but also bounded by the recursion limit.

In [6]:
def is_even(n):
    return True if n == 0 else is_odd(n - 1)

def is_odd(n):
    return False if n == 0 else is_even(n - 1)

print(is_even(10), is_odd(7), is_even(7))   # True True False

True True False


## 7. Interview thinking

**Say:** "The recursion depth can reach n, and Python's default limit is 1000, so I'll convert it to an iterative version using an explicit stack (or a loop, since it is tail recursive)."

**Common mistakes:** raising `setrecursionlimit` to millions (segfault), pushing children in the wrong order (traversal order changes), forgetting to include the **state** (index, path, accumulated value) on the stack, assuming Python optimizes tail calls.

## Quick Revision

- Python: ~1000 frames, no tail-call optimization.
- Tail/linear recursion → loop. Two-term → two variables.
- General: explicit stack of pending work (push children reversed).
- Trampoline = return thunks, loop calls them.
- `lru_cache` for memo; bottom-up for deep problems.